# 군집화 과제 — 와인 데이터

**제출** GitHub `develop` 브랜치에 push · **데이터** `sklearn.datasets.load_wine`

---

수업 실습에서는 **정답이 없는** 고객 데이터를 다뤘습니다.
이번 과제는 **정답이 있는** 데이터입니다. 와인 178병이 실제로 **3종류**로 나뉘어 있어요.

정답이 있으면 좋은 게 하나 있습니다. **군집화가 실제로 맞았는지 채점할 수 있다**는 것.
그래서 이번 과제의 진짜 주제는 이겁니다.

> ### 지표가 좋다고 군집화가 잘 된 걸까?

---

| | 내용 |
|---|---|
| **Part 1** | 코드를 읽고 **주석 달기** |
| **Part 2** | 결과를 보고 **해석하기** |
| **Part 3** | 직접 판단하고 **근거 쓰기** |

> 답을 적는 칸은 **마크다운 셀**입니다. 길게 쓸 필요 없습니다. **근거를 숫자로 대면** 충분합니다.

## 0. 준비

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, confusion_matrix, adjusted_rand_score
from scipy.optimize import linear_sum_assignment

import warnings; warnings.filterwarnings('ignore')

wine = load_wine()
X_raw = pd.DataFrame(wine.data, columns=wine.feature_names)
y_true = wine.target                       # 실제 와인 종류 (0, 1, 2)

print(f'와인 {X_raw.shape[0]}병 · 화학 성분 {X_raw.shape[1]}개 · 실제 종류 {len(set(y_true))}가지')
X_raw.head()

와인 178병 · 화학 성분 13개 · 실제 종류 3가지


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


군집화 결과를 **정답과 맞춰보는** 함수입니다. 그대로 실행하시면 됩니다.

군집 번호(0,1,2)와 정답 번호(0,1,2)는 **순서가 다를 수 있어서**, 가장 잘 맞는 짝을 찾아 채점합니다.

In [2]:
def match_accuracy(y_true, labels):
    """군집 번호와 정답 번호를 최적으로 짝지어 일치율을 계산"""
    mask = labels != -1                       # 노이즈(-1)는 오답 처리
    if mask.sum() == 0:
        return 0.0
    cm = confusion_matrix(y_true[mask], labels[mask])
    row, col = linear_sum_assignment(-cm)
    return cm[row, col].sum() / len(y_true)

---
# Part 1. 코드를 읽고 주석 달기

아래는 **군집화 파이프라인 전체**입니다. 지금은 주석이 없습니다.

In [4]:
X_scaled = StandardScaler().fit_transform(X_raw)                          # ①

pca = PCA(n_components=2)                                                # ②
X_pca = pca.fit_transform(X_scaled)                                       # ③

kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)  # ④
labels = kmeans.fit_predict(X_scaled)                                     # ⑤

sil = silhouette_score(X_scaled, labels)                                  # ⑥
acc = match_accuracy(y_true, labels)                                      # ⑦

print(f'실루엣 {sil:.3f} · 정답 일치 {acc:.3f}')

실루엣 0.285 · 정답 일치 0.966


### 1-1. 각 줄이 하는 일을 채우세요

`무엇을 하는가`뿐 아니라 **`왜 필요한가`** 를 꼭 적어주세요. 이게 핵심입니다.

| 줄 | 무엇을 하는가 | 왜 필요한가 |
|---|---|---|
| ① | `X_raw`의 각 변수의 평균을 0, 표준편차를 1이 되도록 표준화한다. | 변수마다 측정 단위와 값의 범위가 다르기 때문에, 특정 변수의 크기가 거리 계산에 과도하게 영향을 주는 것을 막기 위해 필요하다. |
| ② | PCA를 사용하여 데이터를 2개의 주성분으로 축소하는 객체를 만든다. | 13개의 변수를 2개의 차원으로 줄여 데이터를 시각적으로 표현하고 주요 변동을 파악하기 위해 필요하다. |
| ③ | 표준화된 데이터를 PCA에 적용하여 2차원 데이터로 변환한다. | 13차원의 데이터를 2개의 주성분으로 변환하여 저차원 공간에서 데이터를 표현하기 위해 필요하다. |
| ④ | 3개의 군집을 만드는 K-means 모델을 설정한다. | 실제 와인이 3종류라는 것을 알고 있으므로 3개의 군집으로 나누고, `random_state`를 고정하여 실행할 때마다 같은 결과를 얻기 위해 필요하다. |
| ⑤ | `X_scaled`를 K-means에 입력하여 각 데이터에 군집 번호를 부여한다. | 표준화된 원본 13개 변수의 정보를 이용하여 실제 군집화를 수행하기 위해 필요하다. |
| ⑥ | 군집 내 데이터가 서로 가깝고 군집 간 데이터가 얼마나 떨어져 있는지를 나타내는 실루엣 점수를 계산한다. | 정답 레이블 없이도 군집의 응집도와 분리도를 평가하기 위해 필요하다. |
| ⑦ | 군집 결과와 실제 와인 종류를 비교하여 정답 일치율을 계산한다. | 실제 정답 레이블과 비교하여 군집화가 실제 와인 종류를 얼마나 잘 구분했는지 확인하기 위해 필요하다. |

### 1-2. 순서에 대한 질문

**(a)** 표준화하지 않은 `X_raw`에 PCA를 먼저 적용하면 어떻게 될까요? 왜 그런가요?

<br> 표준화하지 않은 `X_raw`에 PCA를 먼저 적용하면, 변수의 측정 단위와 분산 크기에 따라 일부 변수가 PCA 결과에 지나치게 큰 영향을 줄 수 있다. 따라서 각 변수의 영향력을 비슷하게 만들기 위해 표준화를 먼저 수행하는 것이 필요하다.

**(b)** ⑤를 보면 군집화에 `X_pca` 가 아니라 **`X_scaled` 를 넣었습니다.** ③에서 PCA를 해놓고 왜 안 썼을까요?

<br> PCA는 데이터를 2차원으로 축소하기 위한 것이고, 실제 군집화에서는 13개의 변수에 포함된 정보를 최대한 활용하기 위해 `X_scaled`를 사용했다. 즉, PCA 결과인 `X_pca`는 주로 차원 축소와 시각화 및 비교를 위해 사용하고, 군집화에는 원래 변수 정보를 유지한 `X_scaled`를 사용한 것이다.

**(c)** ④의 `random_state=42` 를 지우면 무슨 일이 생기나요? k-means의 어떤 성질 때문인가요?

<br> `random_state=42`를 지우면 K-means의 초기 중심값이 달라질 수 있어 실행할 때마다 군집 결과가 달라질 수 있다. K-means는 초기 중심값에 따라 최종 군집 결과가 달라질 수 있기 때문이다.

---
# Part 2. 결과를 보고 해석하기

## 2-1. 차원을 줄였더니 점수가 올랐습니다

In [5]:
rows = []
labels_by_space = {}
for name, data in [('13차원 (원본)', X_scaled), ('2차원 (PCA)', X_pca)]:
    lab = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(data)
    labels_by_space[name] = lab
    rows.append({'데이터': name,
                 '실루엣': round(silhouette_score(data, lab), 3),
                 '정답 일치율': round(match_accuracy(y_true, lab), 3)})

labels_13d = labels_by_space['13차원 (원본)']
labels_2d = labels_by_space['2차원 (PCA)']
cm_between = confusion_matrix(labels_13d, labels_2d)
row, col = linear_sum_assignment(-cm_between)
n_different = len(labels_13d) - cm_between[row, col].sum()
ari_between = adjusted_rand_score(labels_13d, labels_2d)

print(f'두 군집 결과 간 ARI: {ari_between:.3f}')
print(f'군집 번호를 최적으로 맞춘 뒤 배정이 다른 와인: {n_different}병')
pd.DataFrame(rows)

두 군집 결과 간 ARI: 0.932
군집 번호를 최적으로 맞춘 뒤 배정이 다른 와인: 4병


,데이터,실루엣,정답 일치율
0,13차원 (원본),0.285,0.966
1,2차원 (PCA),0.561,0.966


### 2-1

실루엣은 **0.285 → 0.561로 거의 2배**가 되었고, 정답 일치율은 두 경우 모두 96.6%입니다.

ARI(조정 랜드 지수)는 군집 번호와 무관하게 두 군집 결과가 얼마나 같은지 재며, **1이면 완전히 같은 분할**을 뜻합니다.

**(a)** 정답 일치율이 같다는 사실만으로 두 군집 결과가 완전히 같다고 할 수 있을까요? 위에 출력된 ARI와 배정이 다른 와인 수를 근거로 설명하세요.

<br> 정답 일치율이 같더라도 두 군집 결과가 완전히 같다고 할 수 없다. 두 경우의 정답 일치율은 모두 0.966이지만, ARI가 약 0.932로 1이 아니며 군집 번호를 최적으로 맞춘 후에도 4병의 군집 배정이 달랐다. 따라서 두 결과는 매우 유사하지만 완전히 동일한 군집 결과는 아니다.

**(b)** 정답 일치율은 같은데 실루엣은 크게 오른 이유는 무엇일까요?
*(힌트: 실루엣은 무엇을 재나요? 그리고 강의 `2-4`의 **차원의 저주**에서 거리는 어떻게 되었나요?)*

<br> 실루엣은 같은 군집 안의 데이터가 얼마나 가깝고 다른 군집의 데이터와 얼마나 떨어져 있는지를 거리로 평가한다. 13차원에서는 차원이 높아지면서 데이터 간 거리가 전반적으로 비슷해지는 차원의 저주가 나타날 수 있어 군집의 분리가 잘 드러나지 않는다. PCA를 통해 2차원으로 축소하면 상대적으로 군집 간 거리 구조가 더 뚜렷해져 실루엣이 0.285에서 0.561로 증가할 수 있다.

**(c)** 그렇다면 **"실루엣 0.5 이상이면 좋은 군집화"** 같은 절대 기준을 쓸 수 있을까요?

<br> 실루엣 점수는 데이터의 차원과 표현 공간, 거리 구조에 영향을 받기 때문에 0.5 이상이라는 절대적인 기준만으로 군집화의 품질을 판단할 수 없다.

## 2-2. 실루엣 0.285는 나쁜 걸까요?

수업 실습(고객 데이터)에서는 실루엣이 **0.481**이었습니다.
지금 와인 13차원은 **0.285** 로 훨씬 낮습니다. 그런데 정답은 **96.6%** 맞혔어요.

### 2-2

**(a)** 실루엣 0.285를 보고 *"군집화에 실패했다"* 고 판단하면 무엇을 놓치는 건가요?

<br> 실루엣 0.285만 보고 군집화에 실패했다고 판단하면 실제 정답과 얼마나 일치하는지를 놓치게 된다. 이 데이터에서는 실루엣이 0.285로 낮지만 정답 일치율은 96.6%로 매우 높기 때문에, 실루엣만으로 군집화 성능을 판단해서는 안 된다.

**(b)** 정답 레이블이 **없는** 상황(수업의 고객 데이터)에서는 이 함정을 어떻게 피할 수 있을까요?
*(힌트: 실습 4에서 k=3과 k=4 중 무엇을 보고 골랐나요?)*

<br> 정답 레이블이 없다면 실루엣 점수 하나만 보는 것이 아니라 여러 군집 수에 대한 실루엣 점수를 비교하고, 군집의 크기와 분리 정도 및 군집의 해석 가능성을 함께 확인해야 한다. 특히 k=3과 k=4처럼 여러 k를 비교하여 상대적으로 적절한 군집 수를 선택해야 한다.

## 2-3. DBSCAN이 최고 점수를 받았습니다

`eps` 를 바꿔가며 DBSCAN을 돌리고, **노이즈를 제외하고** 실루엣을 계산했습니다.

In [6]:
rows = []
for eps in [0.3, 0.4, 0.5, 0.6, 0.8]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(X_pca)
    n_cluster = len(set(lab)) - (1 if -1 in lab else 0)
    mask = lab != -1
    sil = silhouette_score(X_pca[mask], lab[mask]) if n_cluster > 1 else np.nan
    rows.append({'eps': eps, '군집 수': n_cluster,
                 '노이즈': (~mask).sum(),
                 '노이즈 비율(%)': round((~mask).mean()*100, 1),
                 '실루엣(노이즈 제외)': round(sil, 3),
                 '정답 일치율': round(match_accuracy(y_true, lab), 3)})
pd.DataFrame(rows)

,eps,군집 수,노이즈,노이즈 비율(%),실루엣(노이즈 제외),정답 일치율
0,0.3,5,115,64.6,0.790,0.275
1,0.4,9,60,33.7,0.498,0.404
2,0.5,5,31,17.4,0.523,0.573
3,0.6,2,16,9.0,0.499,0.607
4,0.8,2,5,2.8,0.478,0.635


### 2-3

`eps=0.3` 일 때 실루엣이 **0.790** 으로 표에서 가장 높습니다. Part 2-1의 k-means(0.561)보다도 높아요.

**(a)** 그런데 이 설정은 **178병 중 115병(64.6%)을 노이즈로 버렸습니다.**
노이즈를 제외하고 실루엣을 계산하면 왜 점수가 높게 나올까요?

<br> eps=0.3에서는 전체 178병 중 115병을 노이즈로 제외하고 나머지 데이터만 대상으로 실루엣을 계산한다. 따라서 상대적으로 군집이 잘 분리된 일부 데이터만 평가 대상에 포함되어 실루엣 점수가 높게 나타날 수 있다. 많은 데이터를 노이즈로 제외했기 때문에 높은 실루엣 점수가 전체 데이터의 군집화가 잘 되었다는 의미는 아니다.

**(b)** 정답 일치율 컬럼을 함께 보세요. 실루엣 1등과 정답 일치율 1등이 같은가요?

<br> 실루엣 1등은 eps=0.3의 0.790이지만 정답 일치율은 0.275에 불과하다. 반면 정답 일치율은 eps=0.8에서 0.635로 가장 높다. 따라서 실루엣이 가장 높은 설정과 실제 정답과 가장 많이 일치하는 설정은 다르다.

**(c)** 이 표를 근거로, **DBSCAN의 실루엣을 k-means의 실루엣과 나란히 비교하면 안 되는 이유**를 한 문장으로 쓰세요.

<br> DBSCAN은 노이즈로 판단한 데이터를 제외하고 실루엣을 계산했기 때문에, 전체 데이터를 대상으로 계산한 K-means의 실루엣과 동일한 조건에서 비교할 수 없다.

---
# Part 3. 직접 판단하고 근거 쓰기

이제 **여러분이 정하세요.** 정답을 맞히는 게 아니라 **근거를 대는 것**이 이 파트의 전부입니다.

## 3-1. k를 정하세요

In [7]:
rows = []
for k in range(2, 7):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    rows.append({'k': k,
                 '실루엣': round(silhouette_score(X_scaled, lab), 3),
                 '정답 일치율': round(match_accuracy(y_true, lab), 3)})
pd.DataFrame(rows)

,k,실루엣,정답 일치율
0,2,0.259,0.601
1,3,0.285,0.966
2,4,0.260,0.831
3,5,0.202,0.657
4,6,0.237,0.713


### 3-1

**(a)** 실루엣만 보고 k를 고른다면 몇인가요? 정답 일치율만 보면 몇인가요?

<br> k=3. 정답 일치율을 기준으로 보아도 k=3이 가장 높다.

**(b)** 만약 **정답 레이블이 없었다면** — 즉 오른쪽 컬럼을 볼 수 없었다면 — 여러분은 k를 몇으로 정했을까요?
그 판단이 결과적으로 맞았나요?

<br> 정답 레이블이 없었다면 실루엣을 기준으로 k=3을 선택했을 것이다. k=2~6 중 k=3의 실루엣이 0.285로 가장 높기 때문이다. 결과적으로 실제 와인 종류도 3가지이므로 이 판단은 실제 정답과 일치했다.

## 3-2. 알고리즘을 고르세요

아래 셀의 `my_model` 을 직접 바꿔가며 최소 **3가지**를 시도해보세요.

In [10]:
# 여기를 바꿔가며 실행하세요
my_model = KMeans(n_clusters=3, n_init=10, random_state=42)

# 시도해볼 것들:
#   GaussianMixture(n_components=3, random_state=42)
#   AgglomerativeClustering(n_clusters=3, linkage='ward')
#   AgglomerativeClustering(n_clusters=3, linkage='single')
#   DBSCAN(eps=2.2, min_samples=5)
#     ※ 2-3에서는 2차원(X_pca)에 eps=0.6을 썼지만 여기는 13차원(X_scaled)입니다.
#        차원이 높아지면 점들 사이 거리가 전반적으로 멀어져서 eps도 훨씬 크게 잡아야 해요.
#        (eps=0.6을 그대로 넣으면 178병이 전부 노이즈가 됩니다. 직접 확인해 보세요.)

labels = my_model.fit_predict(X_scaled)
mask = labels != -1
n_cluster = len(set(labels)) - (1 if -1 in labels else 0)

print(f'모델          : {type(my_model).__name__}')
print(f'군집 수       : {n_cluster}')
print(f'노이즈        : {(~mask).sum()}개')
if n_cluster > 1:
    print(f'실루엣        : {silhouette_score(X_scaled[mask], labels[mask]):.3f}')
else:
    print(f'실루엣        : 계산 불가 — 군집이 {n_cluster}개뿐입니다 (eps를 키워보세요)')
print(f'정답 일치율   : {match_accuracy(y_true, labels):.3f}')

모델          : KMeans
군집 수       : 3
노이즈        : 0개
실루엣        : 0.285
정답 일치율   : 0.966


In [12]:
# 여기를 바꿔가며 실행하세요
my_model = GaussianMixture(n_components=3, random_state=42)

# 시도해볼 것들:
#   GaussianMixture(n_components=3, random_state=42)
#   AgglomerativeClustering(n_clusters=3, linkage='ward')
#   AgglomerativeClustering(n_clusters=3, linkage='single')
#   DBSCAN(eps=2.2, min_samples=5)
#     ※ 2-3에서는 2차원(X_pca)에 eps=0.6을 썼지만 여기는 13차원(X_scaled)입니다.
#        차원이 높아지면 점들 사이 거리가 전반적으로 멀어져서 eps도 훨씬 크게 잡아야 해요.
#        (eps=0.6을 그대로 넣으면 178병이 전부 노이즈가 됩니다. 직접 확인해 보세요.)

labels = my_model.fit_predict(X_scaled)
mask = labels != -1
n_cluster = len(set(labels)) - (1 if -1 in labels else 0)

print(f'모델          : {type(my_model).__name__}')
print(f'군집 수       : {n_cluster}')
print(f'노이즈        : {(~mask).sum()}개')
if n_cluster > 1:
    print(f'실루엣        : {silhouette_score(X_scaled[mask], labels[mask]):.3f}')
else:
    print(f'실루엣        : 계산 불가 — 군집이 {n_cluster}개뿐입니다 (eps를 키워보세요)')
print(f'정답 일치율   : {match_accuracy(y_true, labels):.3f}')

모델          : GaussianMixture
군집 수       : 3
노이즈        : 0개
실루엣        : 0.285
정답 일치율   : 0.966


In [15]:
# 여기를 바꿔가며 실행하세요
my_model = AgglomerativeClustering(n_clusters=3, linkage='ward')

# 시도해볼 것들:
#   GaussianMixture(n_components=3, random_state=42)
#   AgglomerativeClustering(n_clusters=3, linkage='ward')
#   AgglomerativeClustering(n_clusters=3, linkage='single')
#   DBSCAN(eps=2.2, min_samples=5)
#     ※ 2-3에서는 2차원(X_pca)에 eps=0.6을 썼지만 여기는 13차원(X_scaled)입니다.
#        차원이 높아지면 점들 사이 거리가 전반적으로 멀어져서 eps도 훨씬 크게 잡아야 해요.
#        (eps=0.6을 그대로 넣으면 178병이 전부 노이즈가 됩니다. 직접 확인해 보세요.)

labels = my_model.fit_predict(X_scaled)
mask = labels != -1
n_cluster = len(set(labels)) - (1 if -1 in labels else 0)

print(f'모델          : {type(my_model).__name__}')
print(f'군집 수       : {n_cluster}')
print(f'노이즈        : {(~mask).sum()}개')
if n_cluster > 1:
    print(f'실루엣        : {silhouette_score(X_scaled[mask], labels[mask]):.3f}')
else:
    print(f'실루엣        : 계산 불가 — 군집이 {n_cluster}개뿐입니다 (eps를 키워보세요)')
print(f'정답 일치율   : {match_accuracy(y_true, labels):.3f}')

모델          : AgglomerativeClustering
군집 수       : 3
노이즈        : 0개
실루엣        : 0.277
정답 일치율   : 0.927


In [16]:
# 여기를 바꿔가며 실행하세요
my_model = AgglomerativeClustering(n_clusters=3, linkage='single')

# 시도해볼 것들:
#   GaussianMixture(n_components=3, random_state=42)
#   AgglomerativeClustering(n_clusters=3, linkage='ward')
#   AgglomerativeClustering(n_clusters=3, linkage='single')
#   DBSCAN(eps=2.2, min_samples=5)
#     ※ 2-3에서는 2차원(X_pca)에 eps=0.6을 썼지만 여기는 13차원(X_scaled)입니다.
#        차원이 높아지면 점들 사이 거리가 전반적으로 멀어져서 eps도 훨씬 크게 잡아야 해요.
#        (eps=0.6을 그대로 넣으면 178병이 전부 노이즈가 됩니다. 직접 확인해 보세요.)

labels = my_model.fit_predict(X_scaled)
mask = labels != -1
n_cluster = len(set(labels)) - (1 if -1 in labels else 0)

print(f'모델          : {type(my_model).__name__}')
print(f'군집 수       : {n_cluster}')
print(f'노이즈        : {(~mask).sum()}개')
if n_cluster > 1:
    print(f'실루엣        : {silhouette_score(X_scaled[mask], labels[mask]):.3f}')
else:
    print(f'실루엣        : 계산 불가 — 군집이 {n_cluster}개뿐입니다 (eps를 키워보세요)')
print(f'정답 일치율   : {match_accuracy(y_true, labels):.3f}')

모델          : AgglomerativeClustering
군집 수       : 3
노이즈        : 0개
실루엣        : 0.183
정답 일치율   : 0.376


In [13]:
# 여기를 바꿔가며 실행하세요
my_model = DBSCAN(eps=2.2, min_samples=5)

# 시도해볼 것들:
#   GaussianMixture(n_components=3, random_state=42)
#   AgglomerativeClustering(n_clusters=3, linkage='ward')
#   AgglomerativeClustering(n_clusters=3, linkage='single')
#   DBSCAN(eps=2.2, min_samples=5)
#     ※ 2-3에서는 2차원(X_pca)에 eps=0.6을 썼지만 여기는 13차원(X_scaled)입니다.
#        차원이 높아지면 점들 사이 거리가 전반적으로 멀어져서 eps도 훨씬 크게 잡아야 해요.
#        (eps=0.6을 그대로 넣으면 178병이 전부 노이즈가 됩니다. 직접 확인해 보세요.)

labels = my_model.fit_predict(X_scaled)
mask = labels != -1
n_cluster = len(set(labels)) - (1 if -1 in labels else 0)

print(f'모델          : {type(my_model).__name__}')
print(f'군집 수       : {n_cluster}')
print(f'노이즈        : {(~mask).sum()}개')
if n_cluster > 1:
    print(f'실루엣        : {silhouette_score(X_scaled[mask], labels[mask]):.3f}')
else:
    print(f'실루엣        : 계산 불가 — 군집이 {n_cluster}개뿐입니다 (eps를 키워보세요)')
print(f'정답 일치율   : {match_accuracy(y_true, labels):.3f}')

모델          : DBSCAN
군집 수       : 2
노이즈        : 55개
실루엣        : 0.348
정답 일치율   : 0.483


### 3-2

시도한 결과를 채우세요.

| 알고리즘 | 군집 수 | 실루엣 | 정답 일치율 |
|---|---|---|---|
|Kmeans| 3 | 0.285 | 0.966 |
|GaussianMixture| 3 | 0.285 | 0.966 |
| AgglomerativeClustering(ward) | 3 | 0.277 | 0.927 |
| AgglomerativeClustering(single) | 3 | 0.183 | 0.376 |
|DBSCAN| 2 | 0.348 | 0.483 |

**(a)** 가장 좋다고 판단한 알고리즘은 무엇이고, **무엇을 근거로** 골랐나요?

<br> KMeans를 선택하였다. KMeans는 실루엣 0.285와 정답 일치율 0.966을 보였으며, 특히 정답 일치율이 매우 높아 실제 와인 종류를 잘 구분해 이를 근거로 골랐다.

**(b)** `linkage='single'` 을 넣어보셨나요? 결과가 왜 그렇게 나왔는지 **실습 2에서 본 것**과 연결해 설명하세요.

<br> `linkage='single'`에서는 실루엣이 0.183, 정답 일치율이 0.376으로 다른 방법보다 낮게 나타났다. Single linkage는 두 군집 사이에서 가장 가까운 두 데이터 간 거리를 기준으로 군집을 연결하기 때문에, 가까운 데이터가 연쇄적으로 연결되는 chaining 현상이 발생할 수 있다. 이로 인해 군집이 길게 연결되거나 서로 다른 군집이 합쳐지는 문제가 나타날 수 있다.

## 3-3. 마지막 — 군집에 이름 붙이기

In [14]:
# 3-2에서 가장 좋다고 판단한 모델로 바꾸세요.
# 아래에는 KMeans를 기본값으로 두었습니다.
best_model = KMeans(n_clusters=3, n_init=10, random_state=42)
best_labels = best_model.fit_predict(X_scaled)

t = X_raw.copy(); t['군집'] = best_labels
t.groupby('군집')[['alcohol', 'flavanoids', 'color_intensity', 'proline', 'hue']].mean().round(2)

,alcohol,flavanoids,color_intensity,proline,hue
군집,,,,,
0,12.25,2.05,2.97,510.17,1.06
1,13.13,0.82,7.23,619.06,0.69
2,13.68,3.00,5.45,1100.23,1.07


### 3-3

**(a)** 3-2에서 선택한 모델의 결과로 나온 세 군집의 특징을 한 줄씩 쓰고, **이름을 붙여보세요.**

| 군집 | 특징 | 이름 |
|---|---|---|
| 0 | alcohol, proline, color_intensity가 상대적으로 낮고 flavanoids가 중간 수준이다. | 저알코올·저프로린형 |
| 1 | flavanoids와 hue가 낮고 color_intensity가 높다. | 고색도·저플라바노이드형 |
| 2 | alcohol, flavanoids, proline이 높고 total_phenols도 높다. | 고알코올·고프로린형 |

<br>

**(b)** 수업 실습에서는 *"군집에 이름을 붙일 수 없다면 잘못 나눈 것"* 이라고 했습니다.
와인 데이터에서 이름을 붙이는 것과, 고객 데이터에서 이름을 붙이는 것은 **무엇이 다른가요?**

<br> 와인 데이터에서는 정답 레이블이 있기 때문에 군집의 특징을 실제 와인 종류와 비교하여 확인할 수 있다. 반면 고객 데이터에서는 정답 레이블이 없기 때문에 군집별 고객의 소비 패턴이나 특성을 분석하여 사람이 이해할 수 있는 의미를 부여해야 한다. 따라서 고객 데이터에서는 군집의 특징을 바탕으로 직접 이름을 붙이고 그 이름이 데이터의 특성을 잘 설명하는지 판단해야 한다.

---
<div style="padding:16px;border-radius:8px">

### 이 과제가 물어본 것

처음의 질문으로 돌아갑니다.

> **지표가 좋다고 군집화가 잘 된 걸까?**

- **2-1** — PCA 전후의 정답 일치율이 같아도 군집 배정은 완전히 같지 않으며, 실루엣은 표현 공간에 따라 크게 바뀌 수 있습니다.
- **2-2** — 실루엣 0.285가 정답 96.6%를 맞혔습니다.
- **2-3** — 데이터의 3분의 2를 버리면 실루엣이 최고가 됩니다.

**지표는 후보를 좁혀줄 뿐입니다. 최종 판단은 여러분이 합니다.**

</div>

수고하셨습니다!